# Burgers' equation in 1D

## Setting up the problem

We solve the one-dimensional Burgers' equation with periodic boundary conditions, i.e.
$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = D \frac{\partial^2 u}{\partial x^2},\quad \text{for}\quad x \in (0,L)\quad \text{and}\quad t > 0,$$
$$u(0,t) = u(L,t),$$
$$u(x,0) = g(x),$$
where:
-  $x$ is the spatial coordinate,
-  $t$ is the time,
-  $D$ is the diffusion coefficient,
-  $L$ is the length of the domain,
-  $u(x,t)$ is the variable of interest,
-  $g(x)$ is the initial profile.

In [16]:
using Mantis
using GLMakie
using LinearAlgebra

# Diffusion coefficient
D = 0.01

# Length of the domain
L = 1.0

# Initial profile
g_expression(x) = [@. exp(-(x[:, 1] - L/2)^2 / (2 * (L/4)^2))];

## Defining the finite element space

Now, we choose the number of elements to divide the domain into, and the polynomial degree and smoothness of the basis functions, to create a piecewise-polynomial function space $V_h$.

In [17]:
# Number of elements
num_elements = 25

# Polynomial degree of the basis functions
polynomial_degree = 2

# Smoothness of the basis functions (-1 <= smoothness <= p-1)
smoothness = 1

# Create a piecewise-polynomial function space
geometry = Geometry.create_cartesian_box((0.0,), (L,), (num_elements,))
V_h = FunctionSpaces.BSplineSpace(geometry, polynomial_degree, smoothness)

# Use this function space as a differential form
V⁰ = Forms.FormSpace(0, V_h, L"V^0")

# Use the expression as a form field
g = Forms.AnalyticalFormField(0, g_expression, geometry, "g");

## Deriving the weak form and assembling the system of ODEs

We define the trial and test space, respectively, as
$$S_h := \{v_h(x) \in V_h\ :\ v_h(0) = v_h(L)\},$$
$$W_h := S_h.$$

The weak problem is then: find $u_h(x,t) \in S_h$ at fixed time $t$ such that
$$\int_0^L \frac{\partial u_h}{\partial t} w_h dx + \int_0^L u_h \frac{\partial u_h}{\partial x} w_h dx = D \int_0^L \frac{\partial^2 u_h}{\partial x^2} w_h dx,\quad \forall w_h \in W_h$$
$$\iff \int_0^L \frac{\partial u_h}{\partial t} w_h dx - D \int_0^L \frac{\partial^2 u_h}{\partial x^2} w_h dx + \int_0^L \frac{\partial}{\partial x} (\frac{1}{2} u_h^2) w_h dx = 0,\quad \forall w_h \in W_h.$$

Using integration by parts and the periodic boundary conditions of $u_h$ and $w_h$ gives
$$\int_0^L \frac{\partial u_h}{\partial t} w_h dx + D \int_0^L \frac{\partial u_h}{\partial x} \frac{\partial w_h}{\partial x} dx - \frac{1}{2} \int_0^L u_h^2 \frac{\partial w_h}{\partial x} dx = 0,\quad \forall w_h \in W_h.$$

Let $\phi_i(x),\ i = 1,\ldots,n$ form a basis for $V_h$. Then we can write $u_h(x,t) = c_1(t) \bar{\phi}(x) + \sum_{j=2}^{n-1} c_j(t) \phi_j(x)$, with $\bar{\phi} = \phi_1 + \phi_n$, because only $\phi_1$ and $\phi_n$ are non-zero and equal to $1$ in respectively $x = 0$ and $x = L$. Since we can do the same for all $w_h \in W_h$, the following $n - 1$ equations must be satisfied:
$$\int_0^L \frac{\partial}{\partial t} (c_1 \bar{\phi} + \sum_{j=2}^{n-1} c_j \phi_j) \bar{\phi} dx + D \int_0^L \frac{\partial}{\partial x} (c_1 \bar{\phi} + \sum_{j=2}^{n-1} c_j \phi_j) \frac{d\bar{\phi}}{dx} dx - \frac{1}{2} \int_0^L u_h^2 \frac{d\bar{\phi}}{dx} dx = 0,$$
$$\int_0^L \frac{\partial}{\partial t} (c_1 \bar{\phi} + \sum_{j=2}^{n-1} c_j \phi_j) \phi_i dx + D \int_0^L \frac{\partial}{\partial x} (c_1 \bar{\phi} + \sum_{j=2}^{n-1} c_j \phi_j) \frac{d\phi_i}{dx} dx - \frac{1}{2} \int_0^L u_h^2 \frac{d\phi_i}{dx} dx = 0,\quad i = 2,\ldots,n-1$$
$$\iff \frac{dc_1}{dt} \int_0^L \bar{\phi} \bar{\phi} dx + \sum_{j=2}^{n-1} \frac{dc_j}{dt} \int_0^L \phi_j \bar{\phi} dx + D c_1 \int_0^L \frac{d\bar{\phi}}{dx} \frac{d\bar{\phi}}{dx} dx + D \sum_{j=2}^{n-1} c_j \int_0^L \frac{d\phi_j}{dx} \frac{d\bar{\phi}}{dx} dx - \frac{1}{2} \int_0^L u_h^2 \frac{d\bar{\phi}}{dx} dx = 0,$$
$$\frac{dc_1}{dt} \int_0^L \bar{\phi} \phi_i dx + \sum_{j=2}^{n-1} \frac{dc_j}{dt} \int_0^L \phi_j \phi_i dx + D c_1 \int_0^L \frac{d\bar{\phi}}{dx} \frac{d\phi_i}{dx} dx + D \sum_{j=2}^{n-1} c_j \int_0^L \frac{d\phi_j}{dx} \frac{d\phi_i}{dx} dx - \frac{1}{2} \int_0^L u_h^2 \frac{d\phi_i}{dx} dx = 0,\quad i = 2,\ldots,n-1.$$

We can write this system of equations as
$$\mathbf{M} \frac{d\mathbf{C}}{dt} + \mathbf{KC} + \mathbf{F}(u_h) = \mathbf{0},$$
with:
-  $\mathbf{M}_{11} = \int_0^L \phi_1 \phi_1 dx + \int_0^L \phi_n \phi_1 dx + \int_0^L \phi_1 \phi_n dx + \int_0^L \phi_n \phi_n dx,$
-  $\mathbf{M}_{i1} = \int_0^L \phi_1 \phi_i dx + \int_0^L \phi_n \phi_i dx,\quad i = 2,\ldots,n-1,$
-  $\mathbf{M}_{1j} = \int_0^L \phi_j \phi_1 dx + \int_0^L \phi_j \phi_n dx,\quad j = 2,\ldots,n-1,$
-  $\mathbf{M}_{ij} = \int_0^L \phi_j \phi_i dx,\quad i, j = 2,\ldots,n-1,$
-  $\mathbf{K}_{11} = D \int_0^L \frac{d\phi_1}{dx} \frac{d\phi_1}{dx} dx + D \int_0^L \frac{d\phi_n}{dx} \frac{d\phi_1}{dx} dx + D \int_0^L \frac{d\phi_1}{dx} \frac{d\phi_n}{dx} dx + D \int_0^L \frac{d\phi_n}{dx} \frac{d\phi_n}{dx} dx,$
-  $\mathbf{K}_{i1} = D \int_0^L \frac{d\phi_1}{dx} \frac{d\phi_i}{dx} dx + D \int_0^L \frac{d\phi_n}{dx} \frac{d\phi_i}{dx} dx,\quad i = 2,\ldots,n-1,$
-  $\mathbf{K}_{1j} = D \int_0^L \frac{d\phi_j}{dx} \frac{d\phi_1}{dx} dx + D \int_0^L \frac{d\phi_j}{dx} \frac{d\phi_n}{dx} dx,\quad j = 2,\ldots,n-1,$
-  $\mathbf{K}_{ij} = D \int_0^L \frac{d\phi_j}{dx} \frac{d\phi_i}{dx} dx,\quad i, j = 2,\ldots,n-1,$
-  $\mathbf{F}_1(u_h) = -\frac{1}{2} \int_0^L u_h^2 \frac{d\phi_1}{dx} dx - \frac{1}{2} \int_0^L u_h^2 \frac{d\phi_n}{dx} dx,$
-  $\mathbf{F}_i(u_h) = -\frac{1}{2} \int_0^L u_h^2 \frac{d\phi_i}{dx} dx,\quad i = 2,\ldots,n-1.$

In [18]:
# Function to assemble the system matrices
function assemble_system_matrices(
    V::Forms.FormSpace,
    D::Float64,
    dΩ::Quadrature.AbstractQuadratureRule,
)
    
    weak_form_inputs = Assemblers.WeakFormInputs(V, V)

    w⁰ = Assemblers.get_test_form(weak_form_inputs)
    u⁰ = Assemblers.get_trial_form(weak_form_inputs)

    M = ∫(w⁰ ∧ ★(u⁰), dΩ)
    lhs_expressions = ((M,),)

    rhs_expressions = ((0,),)

    weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
    M, _ = Assemblers.assemble(weak_form)

    K = ∫(d(w⁰) ∧ ★(d(u⁰)), dΩ)
    lhs_expressions = ((K,),)

    weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
    K, _ = Assemblers.assemble(weak_form)

    return M, D.*K
end

# Define the quadrature
quadrature_degree = polynomial_degree + 2
quadrature_rule = Quadrature.gauss_legendre(quadrature_degree)
dΩ = Quadrature.StandardQuadrature(quadrature_rule, num_elements)

# Assemble the system matrices
M_full, K_full = assemble_system_matrices(V⁰, D, dΩ)

# Handle periodic boundary conditions
num_basis = Forms.get_num_basis(V⁰)
M = zeros(num_basis-1, num_basis-1)
K = zeros(num_basis-1, num_basis-1)

for i in 1:(num_basis-1)
    M[i, 1] += M_full[i, 1] + M_full[i, num_basis]
    M[1, i] += M_full[1, i] + M_full[num_basis, i]

    K[i, 1] += K_full[i, 1] + K_full[i, num_basis]
    K[1, i] += K_full[1, i] + K_full[num_basis, i]
end

M[2:end, 2:end] = M_full[2:(end-1), 2:(end-1)]
K[2:end, 2:end] = K_full[2:(end-1), 2:(end-1)];

## Time integration with Newton-Raphson

$$\mathbf{M}\frac{d\mathbf{C}}{dt}+\mathbf{KC}+\mathbf{F}(u_h)=\mathbf{0}$$

$$\mathbf{M}\frac{\mathbf{C}^{n+1}-\mathbf{C}^n}{\Delta t}+\mathbf{K}(\frac{\mathbf{C}^{n+1}+\mathbf{C}^n}{2})+\bar{\mathbf{F}}=\mathbf{0}$$

$u_h^{n}= \mathbf{C}_1^n \bar{\phi} + \sum_{j=2}^{n-1} \mathbf{C}_j^n \phi_j$ with $\bar{\phi}=\phi_1+\phi_n$

Midpoint:
-  $\bar{\mathbf{F}}_1 = -\frac{1}{2} \int_0^L \frac{(u_h^{n+1})^2+(u_h^n)^2}{2} \frac{d\phi_1}{dx} dx - \frac{1}{2} \int_0^L \frac{(u_h^{n+1})^2+(u_h^n)^2}{2} \frac{d\phi_n}{dx} dx,$
-  $\bar{\mathbf{F}}_i = -\frac{1}{2} \int_0^L \frac{(u_h^{n+1})^2+(u_h^n)^2}{2} \frac{d\phi_i}{dx} dx,\quad i = 2,\ldots,n-1.$

Poisson:
-  $\bar{\mathbf{F}}_1 = -\frac{1}{2} \int_0^L \frac{(u_h^{n+1})^2+u_h^{n+1}u_h^n+(u_h^n)^2}{3} \frac{d\phi_1}{dx} dx - \frac{1}{2} \int_0^L \frac{(u_h^{n+1})^2+u_h^{n+1}u_h^n+(u_h^n)^2}{3} \frac{d\phi_n}{dx} dx,$
-  $\bar{\mathbf{F}}_i = -\frac{1}{2} \int_0^L \frac{(u_h^{n+1})^2+u_h^{n+1}u_h^n+(u_h^n)^2}{3} \frac{d\phi_i}{dx} dx,\quad i = 2,\ldots,n-1.$

$$\mathbf{N}:=\mathbf{M}(\mathbf{C}^{n+1}-\mathbf{C}^n)+\frac{\Delta t}{2}\mathbf{K}(\mathbf{C}^{n+1}+\mathbf{C}^n)+\Delta t\bar{\mathbf{F}}$$
$$\mathbf{J}_{\mathbf{N}}=[\frac{\partial \mathbf{N}_i}{\partial \mathbf{C}_j^{n+1}}]_{ij}=\mathbf{M}+\frac{\Delta t}{2}\mathbf{K}+\Delta t\mathbf{J}_{\bar{\mathbf{F}}}$$

Midpoint:
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{11}=-\frac{1}{2}\int_0^L u_h^{n+1}\phi_1\frac{d\phi_1}{dx}-\frac{1}{2}\int_0^L u_h^{n+1}\phi_n\frac{d\phi_1}{dx}-\frac{1}{2}\int_0^L u_h^{n+1}\phi_1\frac{d\phi_n}{dx}-\frac{1}{2}\int_0^L u_h^{n+1}\phi_n\frac{d\phi_n}{dx},$
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{i1}=-\frac{1}{2}\int_0^L u_h^{n+1}\phi_1\frac{d\phi_i}{dx} dx-\frac{1}{2}\int_0^L u_h^{n+1}\phi_n\frac{d\phi_i}{dx} dx,\quad i = 2,\ldots,n-1,$
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{1j}=-\frac{1}{2}\int_0^L u_h^{n+1}\phi_j\frac{d\phi_1}{dx} dx-\frac{1}{2}\int_0^L u_h^{n+1}\phi_j\frac{d\phi_n}{dx} dx,\quad j = 2,\ldots,n-1,$
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{ij}=-\frac{1}{2}\int_0^L u_h^{n+1}\phi_j\frac{d\phi_i}{dx} dx,\quad i, j = 2,\ldots,n-1.$

Poisson:
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{11}=-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_1\frac{d\phi_1}{dx}-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_n\frac{d\phi_1}{dx}-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_1\frac{d\phi_n}{dx}-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_n\frac{d\phi_n}{dx},$
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{i1}=-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_1\frac{d\phi_i}{dx} dx-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_n\frac{d\phi_i}{dx} dx,\quad i = 2,\ldots,n-1,$
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{1j}=-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_j\frac{d\phi_1}{dx} dx-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_j\frac{d\phi_n}{dx} dx,\quad j = 2,\ldots,n-1,$
-  $(\mathbf{J}_{\bar{\mathbf{F}}})_{ij}=-\frac{1}{2}\int_0^L \frac{2u_h^{n+1}+u_h^n}{3}\phi_j\frac{d\phi_i}{dx} dx ,\quad i, j = 2,\ldots,n-1.$

Newton-Raphson:
-  Set $x_0 = \mathbf{C}^n$
-  For $k=1,\ldots,K-1$
    -  Solve $\mathbf{J}_{\mathbf{N}}(x^k)\Delta x^{k+1}=\mathbf{N}(x^k)$ for $\Delta x^{k+1}$
    -  Set $x^{k+1} = x^k - \Delta x^{k+1}$
-  Set $\mathbf{C}^{n+1}=x^K$

In [19]:
u_h = Assemblers.solve_L2_projection(V⁰, g, dΩ)
u_h.coefficients[end] = u_h.coefficients[1]

function midpoint_newton_step(V, M, K, u_h, dΩ, dt; tolerance=1e-8, max_iterations=15)

    x = Forms.FormField(V⁰, copy(u_h.coefficients), "x")
    residual_norms = Float64[]

    weak_form_inputs = Assemblers.WeakFormInputs(V, V)

    w⁰ = Assemblers.get_test_form(weak_form_inputs)
    u⁰ = Assemblers.get_trial_form(weak_form_inputs)

    for iteration in 1:max_iterations
        u_mid_coefficients = 0.5 .* (x.coefficients .+ u_h.coefficients)
        # u_mid = Forms.FormField(V⁰, u_mid_coefficients, "u_mid")

        JF_full = -0.5 * ∫((d(w⁰) ∧ u⁰) ∧ x, dΩ)
        lhs_expressions = ((JF_full,),)

        F_full = -0.5 * (0.5 * ∫(d(w⁰) ∧ (u_h ∧ u_h), dΩ) + 0.5 * ∫(d(w⁰) ∧ (x ∧ x), dΩ))
        rhs_expressions = ((F_full,),)

        weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
        JF_full, F_full = Assemblers.assemble(weak_form) #, bc)

        n = size(F_full,1)
        JF = zeros(n-1, n-1)
        F = zeros(n-1)
        for i in 1:(n-1)
            JF[i,1] += JF_full[i,1] + JF_full[i,n]
            JF[1,i] += JF_full[1,i] + JF_full[n,i]
        end
        JF[2:end,2:end] = JF_full[2:end-1,2:end-1]
        F[1] = F_full[1] + F_full[end]
        F[2:end] = F_full[2:end-1]

        N = M * (x.coefficients[1:end-1] .- u_h.coefficients[1:end-1]) + dt .* F + dt .* K * u_mid_coefficients[1:end-1]

        push!(residual_norms, norm(N))
        if residual_norms[end] < tolerance
            break
        end

        J = M .+ dt .* JF .+ (0.5*dt) .* K
        x.coefficients[1:end-1] .-= J \ N
        x.coefficients[end] = x.coefficients[1]

    end

    if residual_norms[end] ≥ tolerance
        @warn "Newton did not converge" residual_norms
    end

    return x, residual_norms
end

function poisson_newton_step(V, M, K, u_h, dΩ, dt; tolerance=1e-8, max_iterations=15)

    x = Forms.FormField(V⁰, copy(u_h.coefficients), "x")
    residual_norms = Float64[]

    weak_form_inputs = Assemblers.WeakFormInputs(V, V)

    w⁰ = Assemblers.get_test_form(weak_form_inputs)
    u⁰ = Assemblers.get_trial_form(weak_form_inputs)

    for iteration in 1:max_iterations
        u_mid_coefficients = 0.5 .* (x.coefficients .+ u_h.coefficients)
        # u_mid = Forms.FormField(V⁰, u_mid_coefficients, "u_mid")

        JF_full = -0.5 * (1/3 * ∫((d(w⁰) ∧ u⁰) ∧ u_h, dΩ) + 2/3 * ∫((d(w⁰) ∧ u⁰) ∧ x, dΩ))
        lhs_expressions = ((JF_full,),)

        F_full = -0.5 * (1/3 * ∫(d(w⁰) ∧ (u_h ∧ u_h), dΩ) + 1/3 * ∫(d(w⁰) ∧ (u_h ∧ x), dΩ) + 1/3 * ∫(d(w⁰) ∧ (x ∧ x), dΩ))
        rhs_expressions = ((F_full,),)

        weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
        JF_full, F_full = Assemblers.assemble(weak_form) #, bc)

        n = size(F_full,1)
        JF = zeros(n-1, n-1)
        F = zeros(n-1)
        for i in 1:(n-1)
            JF[i,1] += JF_full[i,1] + JF_full[i,n]
            JF[1,i] += JF_full[1,i] + JF_full[n,i]
        end
        JF[2:end,2:end] = JF_full[2:end-1,2:end-1]
        F[1] = F_full[1] + F_full[end]
        F[2:end] = F_full[2:end-1]

        N = M * (x.coefficients[1:end-1] .- u_h.coefficients[1:end-1]) + dt .* F + dt .* K * u_mid_coefficients[1:end-1]

        push!(residual_norms, norm(N))
        if residual_norms[end] < tolerance
            break
        end

        J = M .+ dt .* JF .+ (0.5*dt) .* K
        x.coefficients[1:end-1] .-= J \ N
        x.coefficients[end] = x.coefficients[1]

    end

    if residual_norms[end] ≥ tolerance
        @warn "Newton did not converge" residual_norms
    end

    return x, residual_norms
end

# Helper function to easily evaluate our solution
function evaluate_solution(u_h)
    geometry = Forms.get_geometry(u_h)
    num_elements = Geometry.get_num_elements(geometry)
    xi = Points.TensorProductPoints((LinRange(0.0, 1.0, 25),))

    all_x = Vector{Float64}(undef, 25 * num_elements)
    all_u = Vector{Float64}(undef, 25 * num_elements)
    
    for element_id in 1:num_elements
        form_eval, _ = Forms.evaluate(u_h, element_id, xi)
        x = Geometry.evaluate(geometry, element_id, xi)

        all_x[((element_id - 1) * 25 + 1):((element_id) * 25)] = x[:]
        all_u[((element_id - 1) * 25 + 1):((element_id) * 25)] = form_eval[1]
    end

    return all_x, all_u
end

xe, ue = evaluate_solution(u_h)

dt = 0.005
num_steps = 1001
history = (
        t=[0.0],
        newton_residuals=Vector{Float64}[],
    )
snapshots = Dict{Int, NamedTuple}()
snapshot_steps = 0:num_steps
0 in snapshot_steps && (snapshots[0] = (; t=0.0, u_h=Forms.FormField(V⁰, copy(u_h.coefficients), "u")))
for step in 1:num_steps
    u_h, residual_norms = poisson_newton_step(V⁰, M, K, u_h, dΩ, dt)
    push!(history.t, step * dt)
    push!(history.newton_residuals, residual_norms)
    if step in snapshot_steps
        snapshots[step] = (; t=step * dt, u_h=Forms.FormField(V⁰, copy(u_h.coefficients), "u"))
    end
end

fig = Figure()
ax = Axis(
    fig[1, 1];
    xlabel="x",
    ylabel="u",
)
u_plot = Observable(evaluate_solution(snapshots[0].u_h)[2])
lines!(ax, xe, u_plot)
record(
    fig,
    "burgers_1d.mp4",
    0:num_steps;
    framerate=50,
) do step
    u_plot[] = evaluate_solution(snapshots[step].u_h)[2]
end

"burgers_1d.mp4"